# Assignment 6: Grounded Language with CLIP and Pragmatic Reference Games

**EECS 183/283A · Fall 2026 · Due Friday, December 4, 2026 at 11:59 PM PT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zinengtang/cs183-fall26-a6/blob/main/HW6.ipynb)

This homework connects two ideas from the multimodal and pragmatics lectures:

1. **CLIP** (Radford et al., 2021) learns a shared image–text embedding space with a contrastive objective. You will implement the loss and build zero-shot classifiers.
2. **The Rational Speech Acts (RSA) model** (Frank & Goodman, 2012) says that speakers choose utterances that a *literal listener* would interpret correctly, and listeners interpret utterances by reasoning about such speakers. You will implement RSA, then use CLIP as the literal listener in a visual **reference game**.

| Part | Topic | Points |
|---|---|---|
| 1 | CLIP: contrastive loss and zero-shot classification | 35 |
| 2 | RSA with a hand-written lexicon | 30 |
| 3 | Neural RSA: CLIP as the literal listener | 35 |
| 4 | **283A only:** speaking to an unknown partner | 20 |

EECS 183 students do Parts 1–3 (100 points). EECS 283A students do Parts 1–4 (120 points). EECS 183 students may skip Part 4. It is not graded for 183 and gives no extra credit.

**Notes:**

* Keep all code as-is, except where you are explicitly asked to implement something.
* Items marked with a star (★) are written questions. Answer them in the markdown **Your answer** cells. They are graded from the PDF of your notebook.
* Code is autograded from the cells whose first line is `# GRADED:`. The autograder extracts **only** those cells and runs hidden unit tests on the functions in them. So:
  - Do not rename the functions or change their signatures.
  - Do not edit the `# GRADED:` header lines.
  - Keep each graded cell self-contained: the imports it needs go inside the cell, and it must not use variables defined elsewhere in the notebook.
  - Do not load models or data inside graded cells.

### Submission

Upload **two files** to the Gradescope assignment **A6**:

1. `HW6.ipynb`, your notebook (on Colab: *File → Download → Download .ipynb*).
2. `HW6.pdf`, the same notebook with all outputs (*File → Print → Save as PDF*). Check that your plots actually rendered before you export.

If you work locally, `python3 make_submission.py` checks that every graded function is implemented and packs both files into `submission.zip` for you.

### Compute

Everything runs on a free Colab CPU runtime in roughly 10–15 minutes, most of which is downloading models. A GPU (*Runtime → Change runtime type → T4 GPU*) makes it faster but is not needed.

### Collaboration and AI policy

The course policy applies. You may discuss ideas with classmates, but write your own code and answers. List your collaborators here: ______

## Setup

In [ ]:
# Colab ships with everything except possibly `datasets`. Locally: pip install -r requirements.txt
import importlib.util, subprocess, sys
if importlib.util.find_spec("datasets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print("device:", device, "| torch", torch.__version__)

---
# Part 1: CLIP (35 points)

CLIP has an image encoder $f_I$ and a text encoder $f_T$. Given a batch of $N$ matched (image, caption) pairs, it computes unit-normalized embeddings $\mathbf{x}_i = f_I(\text{img}_i)/\lVert f_I(\text{img}_i)\rVert$ and $\mathbf{y}_j = f_T(\text{txt}_j)/\lVert f_T(\text{txt}_j)\rVert$. It then forms the $N\times N$ logit matrix

$$\ell_{ij} = s \cdot \mathbf{x}_i^\top \mathbf{y}_j,$$

where $s = e^{t}$ is a learned **logit scale** (an inverse temperature). The training loss is the symmetric InfoNCE / cross-entropy loss. Each image must pick out its own caption among the $N$ captions in the batch, and each caption must pick out its own image:

$$\mathcal{L} = \frac{1}{2}\left[\frac{1}{N}\sum_{i=1}^N -\log \frac{e^{\ell_{ii}}}{\sum_{j} e^{\ell_{ij}}} \;+\; \frac{1}{N}\sum_{j=1}^N -\log \frac{e^{\ell_{jj}}}{\sum_{i} e^{\ell_{ij}}}\right].$$

## 1.1 The contrastive loss (8 points, autograded)

Implement `clip_contrastive_loss`. Inputs are **not** guaranteed to be normalized. `logit_scale` is the multiplier $s$ itself (for example `100.0`), not $t = \log s$. Use `torch.nn.functional.cross_entropy`. Gradients must flow to the inputs.

In [ ]:
# GRADED: clip_contrastive_loss
import torch
import torch.nn.functional as F


def clip_contrastive_loss(image_emb, text_emb, logit_scale):
    """Symmetric CLIP / InfoNCE loss.

    Args:
        image_emb:   float tensor [N, D], row i is the i-th image embedding (not necessarily unit norm)
        text_emb:    float tensor [N, D], row i is the caption paired with image i
        logit_scale: python float or 0-dim tensor s (the multiplier, NOT log s)
    Returns:
        0-dim tensor: the average of the image->text and text->image cross-entropy losses.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Sanity checks (not graded). Random embeddings at scale 1: loss is about log N. Perfectly aligned pairs at scale 100: loss is about 0.
torch.manual_seed(0)
N, D = 16, 32
a = torch.randn(N, D)
print("random, s=1   :", clip_contrastive_loss(a, torch.randn(N, D), 1.0).item(), " log N =", math.log(N))
print("aligned, s=100:", clip_contrastive_loss(a, 3 * a, 100.0).item())

## Loading CLIP and CIFAR-10

We use the original OpenAI **ViT-B/32** CLIP checkpoint through Hugging Face `transformers`, and a fixed random subset of 1,000 CIFAR-10 test images.

The helper `_as_tensor` makes the code work with both `transformers` 4.x and 5.x. In 5.x, `get_image_features` returns an output object instead of a tensor.

In [ ]:
from transformers import CLIPModel, CLIPProcessor


def load_clip(name):
    model = CLIPModel.from_pretrained(name).to(device).eval()
    processor = CLIPProcessor.from_pretrained(name)
    return model, processor


def _as_tensor(out):
    return out if torch.is_tensor(out) else out.pooler_output


@torch.no_grad()
def encode_images(model, processor, images, batch_size=100):
    feats = []
    for i in range(0, len(images), batch_size):
        x = processor(images=images[i:i + batch_size], return_tensors="pt").to(device)
        feats.append(_as_tensor(model.get_image_features(**x)).float().cpu())
    return torch.cat(feats)


@torch.no_grad()
def encode_texts(model, processor, texts, batch_size=256):
    feats = []
    for i in range(0, len(texts), batch_size):
        x = processor(text=list(texts[i:i + batch_size]), return_tensors="pt", padding=True).to(device)
        feats.append(_as_tensor(model.get_text_features(**x)).float().cpu())
    return torch.cat(feats)


clip_b32, proc_b32 = load_clip("openai/clip-vit-base-patch32")
LOGIT_SCALE = clip_b32.logit_scale.exp().item()
print("learned logit scale s =", round(LOGIT_SCALE, 2))

In [ ]:
from datasets import load_dataset

cifar = load_dataset("uoft-cs/cifar10", split="test")
CLASSES = cifar.features["label"].names
subset = np.random.RandomState(0).choice(len(cifar), 1000, replace=False)
cifar_images = [cifar[int(i)]["img"].convert("RGB") for i in subset]
cifar_labels = np.array([cifar[int(i)]["label"] for i in subset])
print(CLASSES)

t0 = time.time()
cifar_img_emb = encode_images(clip_b32, proc_b32, cifar_images)      # [1000, 512], NOT normalized
print("encoded", tuple(cifar_img_emb.shape), f"in {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img, y in zip(axes, cifar_images, cifar_labels):
    ax.imshow(img); ax.set_title(CLASSES[y], fontsize=9); ax.axis("off")
plt.show()

### ★ Question 1.1: in-batch negatives (4 points, written)

The cell below evaluates your loss on real CLIP embeddings for two batches of 10 (image, caption) pairs. Each caption is `"a photo of a {class}."`.

- **Batch A** has one image from each of the 10 classes.
- **Batch B** has 5 airplanes and 5 ships.

Report both losses. Explain why batch B's loss is much higher even though CLIP classifies these images well. What assumption does the contrastive loss make about the other items in a batch? When does that assumption fail for web-scale (image, alt-text) data?

In [ ]:
caption = lambda y: f"a photo of a {CLASSES[y]}."
batch_a = [int(np.where(cifar_labels == k)[0][0]) for k in range(10)]
batch_b = list(np.where(cifar_labels == 0)[0][:5]) + list(np.where(cifar_labels == 8)[0][:5])
for name, b in [("A (10 distinct classes)", batch_a), ("B (5 airplanes + 5 ships)", batch_b)]:
    txt = encode_texts(clip_b32, proc_b32, [caption(cifar_labels[i]) for i in b])
    print(f"batch {name}: loss = {clip_contrastive_loss(cifar_img_emb[b], txt, LOGIT_SCALE).item():.3f}")

**Your answer (Q1.1):**

*Write your answer here.*

## 1.2 Zero-shot classification (9 points, autograded)

To classify without any training, CLIP embeds one text prompt per class (for example `"a photo of a dog."`) and predicts the class whose text embedding is most similar to the image embedding. **Prompt ensembling** averages the text embeddings of several templates for each class:

1. Embed every template filled in with the class name.
2. **Normalize each embedding**, then average them.
3. Re-normalize the average to unit length.

Implement:

- `build_zero_shot_weights(class_names, templates, encode_text_fn)` → a `[K, D]` tensor of unit-norm class embeddings. `encode_text_fn` maps a list of strings to an (unnormalized) `[n, D]` tensor. Call it **once per class**, with that class's filled-in templates in the order given.
- `zero_shot_logits(image_emb, class_weights, logit_scale)` → `[N, K]` logits $s \cdot \cos(\mathbf{x}_n, \mathbf{w}_k)$. Normalize the image embeddings yourself.

In [ ]:
# GRADED: zero_shot
import torch
import torch.nn.functional as F


def build_zero_shot_weights(class_names, templates, encode_text_fn):
    """
    Args:
        class_names:    list of K strings, e.g. ["airplane", "automobile", ...]
        templates:      list of T format strings containing "{}", e.g. ["a photo of a {}."]
        encode_text_fn: function list[str] -> float tensor [n, D] (unnormalized)
    Returns:
        float tensor [K, D]; row k is the unit-norm (prompt-ensembled) embedding of class k.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def zero_shot_logits(image_emb, class_weights, logit_scale):
    """
    Args:
        image_emb:     float tensor [N, D] (unnormalized)
        class_weights: float tensor [K, D] (unit norm rows, from build_zero_shot_weights)
        logit_scale:   float s
    Returns:
        float tensor [N, K] of logits s * cos(image_n, class_k).
    """
    # YOUR CODE HERE
    raise NotImplementedError

We compare three prompt settings: the bare class name, the single CLIP-paper template, and an 8-template ensemble taken from the CLIP paper's CIFAR prompts.

In [ ]:
TEMPLATE_SETS = {
    "bare name": ["{}"],
    "a photo of a {}.": ["a photo of a {}."],
    "8-template ensemble": [
        "a photo of a {}.", "a blurry photo of a {}.", "a low resolution photo of a {}.",
        "a photo of the small {}.", "a photo of the big {}.", "a pixelated photo of a {}.",
        "a cropped photo of a {}.", "a good photo of a {}.",
    ],
}
encode_text_b32 = lambda texts: encode_texts(clip_b32, proc_b32, texts)

zs_results = {}
for name, templates in TEMPLATE_SETS.items():
    W = build_zero_shot_weights(CLASSES, templates, encode_text_b32)
    logits = zero_shot_logits(cifar_img_emb, W, LOGIT_SCALE)
    zs_results[name] = (logits, (logits.argmax(1).numpy() == cifar_labels).mean())
    print(f"{name:>22s}: zero-shot accuracy = {zs_results[name][1]:.3f}")

### ★ Question 1.2: the logit scale (4 points, written)

The cell below rescales the zero-shot logits of the ensemble classifier to different values of $s$. For each value it reports accuracy, the average probability of the predicted class (confidence), and the expected calibration error (ECE).

(a) Why does accuracy not change with $s$, while confidence does?

(b) Which value of $s$ gives the best-calibrated probabilities? Why might that be different from the $s$ learned in training? Recall that the logits are cosine similarities, which lie in $[-1, 1]$.

(c) CLIP clips its learned logit scale at 100. What would happen to the training loss and its gradients if $s$ were fixed at 1?

In [ ]:
def ece(probs, labels, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    bins = np.linspace(0, 1, n_bins + 1)
    err = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            err += m.mean() * abs((pred[m] == labels[m]).mean() - conf[m].mean())
    return err

cos_ens = zs_results["8-template ensemble"][0] / LOGIT_SCALE
print(f"{'s':>6s} {'acc':>6s} {'conf':>6s} {'ECE':>6s}")
for s in [1, 10, 30, 50, LOGIT_SCALE, 300]:
    p = torch.softmax(s * cos_ens, dim=1).numpy()
    print(f"{s:6.0f} {(p.argmax(1) == cifar_labels).mean():6.3f} {p.max(1).mean():6.3f} {ece(p, cifar_labels):6.3f}")

**Your answer (Q1.2):**

*Write your answer here.*

### ★ Question 1.3: prompts (5 points, written)

(a) Report the three accuracies from the prompt comparison above. Give one reason prompt ensembling helps. Think about what averaging does to the noise in any single text embedding.

(b) Design **your own** template set by editing `my_templates` below. You may use as many templates as you like. Report its accuracy and explain your reasoning in two or three sentences. You get full credit for a thoughtful attempt, even if it does not beat the ensemble.

In [ ]:
my_templates = ["a photo of a {}.", "a close-up photo of a {}."]   # <- edit me
W_mine = build_zero_shot_weights(CLASSES, my_templates, encode_text_b32)
acc_mine = (zero_shot_logits(cifar_img_emb, W_mine, LOGIT_SCALE).argmax(1).numpy() == cifar_labels).mean()
print(f"my templates: zero-shot accuracy = {acc_mine:.3f}")

**Your answer (Q1.3):**

*Write your answer here.*

### ★ Question 1.4: error analysis (5 points, written)

The cell below plots the confusion matrix of the ensemble classifier and shows a few misclassified images.

(a) Name the two most-confused class pairs.

(b) Give a plausible explanation for each pair. Consider the image content, the 32×32 resolution, and what web captions for these concepts tend to say.

(c) Name one limitation of zero-shot classification that this exposes compared with a linear probe trained on CIFAR-10 features.

In [ ]:
pred = zs_results["8-template ensemble"][0].argmax(1).numpy()
cm = np.zeros((10, 10), dtype=int)
for y, p in zip(cifar_labels, pred):
    cm[y, p] += 1

fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

off = cm + cm.T
np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(i + 1, 10)], reverse=True)[:4]
print("most-confused pairs (both directions):", pairs)

wrong = np.where(pred != cifar_labels)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(12, 1.9))
for ax, i in zip(axes, wrong):
    ax.imshow(cifar_images[i]); ax.axis("off")
    ax.set_title(f"{CLASSES[cifar_labels[i]]}\n→ {CLASSES[pred[i]]}", fontsize=8)
plt.show()

**Your answer (Q1.4):**

*Write your answer here.*

---
# Part 2: Rational Speech Acts (30 points)

In a **reference game**, a speaker sees a set of objects and one *target*, and produces an utterance. The listener sees the same objects without knowing which is the target, and must pick it out.

RSA models this as recursive reasoning. Let $[\![u]\!](o)\in\{0,1\}$ be the **lexicon**: whether utterance $u$ is literally true of object $o$. Let $P(o)$ be a prior over objects.

- **Literal listener:** $L_0(o \mid u) \propto [\![u]\!](o)\,P(o)$
- **Pragmatic speaker:** $S_1(u \mid o) \propto \exp\big(\alpha\,(\log L_0(o \mid u) - C(u))\big)$
- **Pragmatic listener:** $L_1(o \mid u) \propto S_1(u \mid o)\,P(o)$

Here $\alpha>0$ is the speaker's rationality and $C(u)\ge 0$ is the cost of an utterance. We use the convention $\log 0 = -\infty$, so $S_1(u\mid o) = 0$ whenever $u$ is false of $o$.

Matrix conventions used throughout:

- `lexicon[u, o]`, `L0[u, o]` and `L1[u, o]` are **[U, O]**, one row per utterance, and each row of a listener sums to 1.
- `S1[o, u]` is **[O, U]**, one row per object, and each row sums to 1.

## 2.1 Implement RSA (15 points, autograded)

Use NumPy. You may assume that every utterance is true of at least one object with nonzero prior, and that every object has at least one true utterance. `cost=None` means all costs are zero. Your `pragmatic_speaker` must not produce `nan`s or warnings when `L0` contains zeros. Subtract the row max before exponentiating, and use `np.errstate(divide="ignore")` around `np.log`.

In [ ]:
# GRADED: rsa
import numpy as np


def literal_listener(lexicon, prior):
    """lexicon: [U, O] array of 0/1 (or nonnegative) truth values; prior: [O]. Returns L0: [U, O], rows sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError


def pragmatic_speaker(L0, alpha=1.0, cost=None):
    """L0: [U, O]; alpha > 0; cost: None or [U]. Returns S1: [O, U], rows sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError


def pragmatic_listener(S1, prior):
    """S1: [O, U]; prior: [O]. Returns L1: [U, O], rows sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError

### The Frank & Goodman (2012) example

Three objects: a blue square, a blue circle, and a green square. Four one-word utterances: *blue*, *green*, *square*, *circle*.

In [ ]:
FG_OBJECTS = ["blue square", "blue circle", "green square"]
FG_UTTS = ["blue", "green", "square", "circle"]
FG_LEXICON = np.array([[1, 1, 0],    # blue
                       [0, 0, 1],    # green
                       [1, 0, 1],    # square
                       [0, 1, 0]])   # circle
uniform = np.ones(3) / 3

L0_fg = literal_listener(FG_LEXICON, uniform)
S1_fg = pragmatic_speaker(L0_fg, alpha=1.0)
L1_fg = pragmatic_listener(S1_fg, uniform)
show = lambda M, rows, cols: display(pd.DataFrame(M, index=rows, columns=cols).round(3))
print("L0(o | u)"); show(L0_fg, FG_UTTS, FG_OBJECTS)
print("S1(u | o)"); show(S1_fg, FG_OBJECTS, FG_UTTS)
print("L1(o | u)"); show(L1_fg, FG_UTTS, FG_OBJECTS)
assert np.allclose(L1_fg[0], [0.6, 0.4, 0.0]), "L1('blue') should be [0.6, 0.4, 0] — check your implementation"
print("✓ L1('blue') = [0.6, 0.4, 0.0]")

### ★ Question 2.1: implicature (5 points, written)

(a) The literal listener is indifferent between the two blue objects given *blue*, but $L_1$ prefers the blue square. Explain this in words, by reasoning about what the speaker *would have said* if the target were the blue circle.

(b) Which utterance does $L_1$ interpret the same way as $L_0$, and why?

**Your answer (Q2.1):**

*Write your answer here.*

### ★ Question 2.2: speaker rationality (5 points, written)

The cell below plots $L_1(\text{blue square} \mid \textit{blue})$ as a function of $\alpha$.

(a) Describe the curve.

(b) What does $L_1$ reduce to as $\alpha \to 0$, and as $\alpha\to\infty$? Justify each limit using the formula for $S_1$.

(c) One more step of recursion gives $S_2(u\mid o) \propto \exp(\alpha \log L_1(o\mid u))$ and $L_2$. Use your functions to compute $L_2(\cdot \mid \textit{blue})$ at $\alpha = 1$ and report it. Does more recursion always sharpen the inference?

In [ ]:
alphas = np.logspace(-1.5, 1.2, 60)
vals = [pragmatic_listener(pragmatic_speaker(L0_fg, a), uniform)[0, 0] for a in alphas]
plt.figure(figsize=(5, 3.2))
plt.semilogx(alphas, vals, color="#2a78d6", lw=2)
plt.axhline(0.5, color="#898781", lw=1, ls="--")
plt.xlabel(r"speaker rationality $\alpha$"); plt.ylabel(r"$L_1$(blue square | 'blue')")
plt.ylim(0.45, 1.02); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# (c) one more level of recursion
# YOUR CODE HERE (not autograded: report what you find in the written answer below)

**Your answer (Q2.2):**

*Write your answer here.*

### ★ Question 2.3: cost and salience (5 points, written)

Now the speaker may also use the two-word utterances *blue square*, *blue circle* and *green square*. These are perfectly informative but cost $c$ each, while one-word utterances cost 0. The cell below plots $S_1(\cdot \mid \text{blue square})$ against $c$ at $\alpha=1$.

(a) At what cost $c$ does the speaker become equally likely to say *blue square* as to say any single one-word utterance? Derive the value analytically, check it against the plot, and explain in one sentence why $c$ is like a per-word "effort" price.

(b) Return to the one-word game with no cost. Make the blue circle **three times** as salient as each of the other objects, with prior $\propto [1, 3, 1]$. Compute $L_1(\cdot\mid\textit{blue})$ and report it. Explain the competition between the prior and the implicature.

In [ ]:
FG2_UTTS = FG_UTTS + ["blue square", "blue circle", "green square"]
FG2_LEXICON = np.vstack([FG_LEXICON, np.eye(3)])
costs = np.linspace(0, 3, 61)
probs = np.array([pragmatic_speaker(literal_listener(FG2_LEXICON, uniform), 1.0,
                                     np.r_[np.zeros(4), np.full(3, c)])[0] for c in costs])
plt.figure(figsize=(5.2, 3.2))
assert np.allclose(probs[:, 0], probs[:, 2])          # 'blue' and 'square' are symmetric here
plt.plot(costs, probs[:, 0], lw=2, color="#2a78d6", label="'blue'  (= 'square')")
plt.plot(costs, probs[:, 4], lw=2, color="#1baf7a", label="'blue square'")
plt.xlabel("cost c of a two-word utterance"); plt.ylabel(r"$S_1$(u | blue square)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# (b) salient blue circle
# YOUR CODE HERE (not autograded: report what you find in the written answer below)

**Your answer (Q2.3):**

*Write your answer here.*

---
# Part 3: Neural RSA, with CLIP as the literal listener (35 points)

A hand-written lexicon does not scale to real images. Instead, we use CLIP's image–text logits $s(i, u)$ as a **soft lexicon**. In a context of $C$ images, the literal listener is

$$L_0(i \mid u) = \frac{\exp s(i,u)}{\sum_{i'=1}^{C} \exp s(i',u)},$$

which is exactly how CLIP does zero-shot *retrieval* among the context images. RSA then builds a speaker and listener on top of it, with no training.

### The world

We render 12 object types (4 colors × 3 shapes), each at 5 random positions and sizes, for 60 images. The vocabulary for Part 3 has seven one-word utterances: four colors and three shapes. Each is turned into a CLIP prompt such as `"a red shape"` or `"a circle"`. (Part 4 adds two-word utterances.)

In [ ]:
COLORS = {"red": (220, 30, 30), "blue": (30, 70, 220), "green": (30, 160, 60), "yellow": (235, 200, 20)}
SHAPES = ["circle", "square", "triangle"]
OBJECTS = [(c, s) for c in COLORS for s in SHAPES]          # 12 object types


def render_shape(color, shape, seed, size=224):
    rng = np.random.RandomState(seed)
    img = Image.new("RGB", (size, size), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    r = rng.randint(45, 70)
    cx, cy = rng.randint(r + 10, size - r - 10, 2)
    fill = COLORS[color]
    if shape == "circle":
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fill)
    elif shape == "square":
        draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=fill)
    else:
        draw.polygon([(cx, cy - r), (cx - r, cy + r * 0.8), (cx + r, cy + r * 0.8)], fill=fill)
    return img


N_RENDERS = 5
shape_images, image_object = [], []
for k, (c, s) in enumerate(OBJECTS):
    for j in range(N_RENDERS):
        shape_images.append(render_shape(c, s, seed=100 * k + j))
        image_object.append(k)
image_object = np.array(image_object)

# vocabulary: 7 one-word utterances (Part 3) + 12 two-word utterances (Part 4)
UTTERANCES = list(COLORS) + SHAPES + [f"{c} {s}" for c, s in OBJECTS]
PROMPT = {u: (f"a {u} shape" if u in COLORS else f"a {u}") for u in UTTERANCES}
ONE_WORD = np.arange(7)                                      # indices of the 7 one-word utterances

fig, axes = plt.subplots(2, 12, figsize=(14, 2.6))
for k in range(12):
    for r in range(2):
        axes[r, k].imshow(shape_images[N_RENDERS * k + r]); axes[r, k].axis("off")
    axes[0, k].set_title(" ".join(OBJECTS[k]), fontsize=7)
plt.show()

We compute one score matrix per CLIP model: `S[i, u]` = logit_scale × cos(image $i$, prompt $u$), with shape `[60 images, 19 utterances]`. We use two models:

- **ViT-B/32**, the model from Part 1, which will act as our speaker's internal listener.
- **ViT-B/16**, a different OpenAI CLIP that was trained on the same data with a different patch size. It plays the role of a listener the speaker has never interacted with.

In [ ]:
@torch.no_grad()
def score_matrix(model, processor, images, texts):
    img = F.normalize(encode_images(model, processor, images), dim=-1)
    txt = F.normalize(encode_texts(model, processor, texts), dim=-1)
    return (model.logit_scale.exp().item() * img @ txt.T).numpy()

prompts = [PROMPT[u] for u in UTTERANCES]
S_b32 = score_matrix(clip_b32, proc_b32, shape_images, prompts)
clip_b16, proc_b16 = load_clip("openai/clip-vit-base-patch16")
S_b16 = score_matrix(clip_b16, proc_b16, shape_images, prompts)
del clip_b16                                                    # free memory

for name, S in [("ViT-B/32", S_b32), ("ViT-B/16", S_b16)]:
    color_acc = (S[:, :4].argmax(1) == image_object // 3).mean()
    shape_acc = (S[:, 4:7].argmax(1) == image_object % 3).mean()
    print(f"{name}: picks the right color word for {color_acc:.0%} of images, right shape word for {shape_acc:.0%}")

### Contexts

Each game has three images, and **index 0 is always the target**. The listener functions are permutation-equivariant, so this costs nothing. We use two kinds of contexts:

- **random:** three distinct object types.
- **hard:** one distractor shares the target's color and the other shares its shape, as in *red circle* vs. *red square* vs. *blue circle*. **No single word is literally true of the target alone**, so the speaker has to rely on the listener reasoning pragmatically.

In [ ]:
def sample_contexts(kind, n, seed=0):
    rng = np.random.RandomState(seed)
    out = []
    while len(out) < n:
        types = rng.choice(len(OBJECTS), 3, replace=False)
        (tc, ts), d1, d2 = [OBJECTS[t] for t in types]
        if kind == "hard" and not ((d1[0] == tc and d2[1] == ts) or (d2[0] == tc and d1[1] == ts)):
            continue
        out.append(np.array([rng.choice(np.where(image_object == t)[0]) for t in types]))
    return out

CONTEXTS = {kind: sample_contexts(kind, 2000, seed=i) for i, kind in enumerate(["random", "hard"])}
print({k: len(v) for k, v in CONTEXTS.items()})

## 3.1 Implement neural RSA (15 points, autograded)

We now work in **log space**, because CLIP logits are large (about 20–35) and probabilities underflow. `scores` is the `[C, U]` slice of a score matrix for one context, with the target at row 0. Implement:

- `context_log_L0(scores)` → `[U, C]`: $\log L_0(i\mid u)$, a log-softmax over the $C$ context images for each utterance.
- `literal_speaker(scores, target)` → `int`. This is the **context-blind** speaker $S_0$: it says the utterance that best describes the target on its own, $\arg\max_u s(\text{target}, u)$. It ignores the distractors.
- `neural_log_S1(log_L0, alpha=1.0, cost=None)` → `[C, U]`: $\log S_1(u \mid i)$, a log-softmax over utterances of $\alpha(\log L_0(i\mid u) - C(u))$.
- `neural_log_L1(log_S1, log_prior=None)` → `[U, C]`: $\log L_1(i\mid u)$, a log-softmax over images of $\log S_1(u\mid i) + \log P(i)$. `log_prior=None` means a uniform prior.

Inputs and outputs are NumPy arrays. Use a numerically stable log-softmax: subtract the max, or use `scipy.special.log_softmax`.

In [ ]:
# GRADED: neural_rsa
import numpy as np


def _log_softmax(x, axis):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))


def context_log_L0(scores):
    """scores: [C, U] CLIP logits for the C context images. Returns log L0: [U, C]."""
    # YOUR CODE HERE
    raise NotImplementedError


def literal_speaker(scores, target=0):
    """Context-blind S0: index of the utterance with the highest score for the target image."""
    # YOUR CODE HERE
    raise NotImplementedError


def neural_log_S1(log_L0, alpha=1.0, cost=None):
    """log_L0: [U, C]. Returns log S1: [C, U]."""
    # YOUR CODE HERE
    raise NotImplementedError


def neural_log_L1(log_S1, log_prior=None):
    """log_S1: [C, U]; log_prior: None or [C]. Returns log L1: [U, C]."""
    # YOUR CODE HERE
    raise NotImplementedError

### Playing reference games

A **speaker** maps a context to an utterance index. A **listener** maps a context and an utterance to the index of its guess. Communication succeeds if the listener picks image 0. The helpers below build speakers and listeners from a score matrix, using your functions.

In [ ]:
def make_S0(S, utts=ONE_WORD):
    return lambda ctx: literal_speaker(S[ctx][:, utts], 0)

def make_S1(S, utts=ONE_WORD, alpha=1.0, cost=None):
    return lambda ctx: int(np.argmax(neural_log_S1(context_log_L0(S[ctx][:, utts]), alpha, cost)[0]))

def make_L0(S, utts=ONE_WORD):
    return lambda ctx, u: int(np.argmax(context_log_L0(S[ctx][:, utts])[u]))

def make_L1(S, utts=ONE_WORD, alpha=1.0):
    return lambda ctx, u: int(np.argmax(neural_log_L1(neural_log_S1(context_log_L0(S[ctx][:, utts]), alpha))[u]))

def play(speaker, listener, contexts):
    """Fraction of games in which the listener picks the target (index 0)."""
    return float(np.mean([listener(ctx, speaker(ctx)) == 0 for ctx in contexts]))

def show_game(ctx, utt_text, guesses=None, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(5.4, 2.1))
    for j, ax in enumerate(axes):
        ax.imshow(shape_images[ctx[j]]); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor("#1baf7a" if j == 0 else "#c3c2b7"); sp.set_linewidth(3 if j == 0 else 1)
        if guesses and j in guesses:
            ax.set_xlabel(guesses[j], fontsize=8)
    axes[0].set_title("target", fontsize=8, color="#1baf7a")
    fig.suptitle(f"{title}  utterance: '{utt_text}'", fontsize=9); plt.tight_layout(); plt.show()

### Experiment A: is a context-aware speaker needed?

The speaker and the listener both use **ViT-B/32**, so the speaker's internal model of the listener is exact. This is "self-play".

In [ ]:
L0_b32 = make_L0(S_b32)
expA = {}
for kind in ["random", "hard"]:
    for sname, spk in [("S0 (context-blind)", make_S0(S_b32)), ("S1 (pragmatic)", make_S1(S_b32))]:
        expA[(kind, sname)] = play(spk, L0_b32, CONTEXTS[kind])
display(pd.Series(expA).unstack().round(3).rename_axis(index="context", columns="speaker → L0 (B/32)"))

# one hard game where S0 fails and S1 succeeds
S0f, S1f = make_S0(S_b32), make_S1(S_b32)
for ctx in CONTEXTS["hard"]:
    u0, u1 = S0f(ctx), S1f(ctx)
    if L0_b32(ctx, u0) != 0 and L0_b32(ctx, u1) == 0:
        show_game(ctx, UTTERANCES[u0], {L0_b32(ctx, u0): "L0 guess"}, "S0:")
        show_game(ctx, UTTERANCES[u1], {L0_b32(ctx, u1): "L0 guess"}, "S1:")
        break

### ★ Question 3.1 (6 points, written)

(a) Report the four numbers from Experiment A.

(b) Explain why $S_0$ does much worse in hard contexts, using the example game above.

(c) In hard contexts, no single word is literally unique to the target, yet $S_1 \to L_0$ still succeeds well above chance (1/3). What property of CLIP's *soft* lexicon is $S_1$ exploiting? Look at `np.exp(context_log_L0(S_b32[ctx][:, ONE_WORD]))` for a couple of hard contexts.

**Your answer (Q3.1):**

*Write your answer here.*

### Experiment B: partner mismatch

The speaker still plans with its internal ViT-B/32 listener, but the **actual** listener is ViT-B/16. The speaker never sees ViT-B/16. As an upper bound, the "oracle" speaker plans with the true listener's model.

In [ ]:
L0_b16 = make_L0(S_b16)
expB = {}
for kind in ["random", "hard"]:
    expB[(kind, "S1(B/32) → L0(B/32)  [self-play]")] = play(make_S1(S_b32), L0_b32, CONTEXTS[kind])
    expB[(kind, "S1(B/32) → L0(B/16)  [new partner]")] = play(make_S1(S_b32), L0_b16, CONTEXTS[kind])
    expB[(kind, "S1(B/16) → L0(B/16)  [oracle]")] = play(make_S1(S_b16), L0_b16, CONTEXTS[kind])
display(pd.Series(expB).unstack(0).round(3))

# where do the two listeners disagree? average L0(target | u) for the utterance S1(B/32) chose, in hard contexts
agree = {"B/32": [], "B/16": []}
for ctx in CONTEXTS["hard"]:
    u = make_S1(S_b32)(ctx)
    agree["B/32"].append(np.exp(context_log_L0(S_b32[ctx][:, ONE_WORD])[u, 0]))
    agree["B/16"].append(np.exp(context_log_L0(S_b16[ctx][:, ONE_WORD])[u, 0]))
plt.figure(figsize=(4.2, 3.6))
plt.scatter(agree["B/32"], agree["B/16"], s=6, alpha=0.35, color="#2a78d6")
plt.plot([0, 1], [0, 1], color="#898781", lw=1, ls="--")
plt.xlabel("L0_B/32(target | u)  — what the speaker expects"); plt.ylabel("L0_B/16(target | u)  — what happens")
plt.title("hard contexts, u = S1(B/32)'s choice", fontsize=9); plt.tight_layout(); plt.show()

### ★ Question 3.2 (7 points, written)

(a) Report the drop from self-play to a new partner in hard contexts. Compare it with the drop in random contexts.

(b) Use the scatter plot to explain *why* hard contexts are fragile. What has the speaker learned to exploit, and why might it not transfer to a different model trained on the same data?

(c) Connect this to self-play in multi-agent RL or emergent communication. Agents trained together often develop conventions that fail with new partners. Name one thing a speaker could do differently to be more robust when it does not know who its listener is. (Part 4 explores this for 283A.)

**Your answer (Q3.2):**

*Write your answer here.*

### Experiment C: pragmatic listening

So far the listener has been literal. Now the listener runs one step of RSA on its **own** model: it computes $L_1$ from its own $S_1$, built from its own $L_0$. We evaluate it with the same speakers.

In [ ]:
expC = {}
for kind in ["random", "hard"]:
    for sname, spk in [("S0(B/32)", make_S0(S_b32)), ("S1(B/32)", make_S1(S_b32))]:
        for lname, lis in [("L0(B/16)", L0_b16), ("L1(B/16)", make_L1(S_b16))]:
            expC[(kind, sname, lname)] = play(spk, lis, CONTEXTS[kind])
display(pd.Series(expC).unstack().round(3))

# alpha sweep for the pragmatic listener in hard contexts
alphas_c = [0.1, 0.3, 1.0, 3.0, 10.0]
plt.figure(figsize=(4.6, 3.2))
for sname, spk, color in [("S0(B/32)", make_S0(S_b32), "#eb6834"), ("S1(B/32)", make_S1(S_b32), "#2a78d6")]:
    plt.semilogx(alphas_c, [play(spk, make_L1(S_b16, alpha=a), CONTEXTS["hard"]) for a in alphas_c],
                 "o-", color=color, label=f"{sname} → L1(B/16)")
    plt.axhline(play(spk, L0_b16, CONTEXTS["hard"]), color=color, ls=":", lw=1)
plt.xlabel(r"listener's assumed speaker rationality $\alpha$"); plt.ylabel("accuracy (hard)")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### ★ Question 3.3 (7 points, written)

(a) Report how accuracy changes when the B/16 listener switches from $L_0$ to $L_1$, for each speaker, in both random and hard contexts. The dotted lines in the plot are the $L_0$ baselines.

(b) In hard contexts, why does a pragmatic listener help so much? Relate your answer to Question 2.1. Which kind of speaker benefits more, and why?

(c) Does pragmatic listening ever *hurt*? Think about what $L_1$ assumes about the speaker, and when that assumption is violated.

**Your answer (Q3.3):**

*Write your answer here.*

---
# Part 4 (EECS 283A only): Speaking to an unknown partner (20 points)

In Experiment B the speaker overfit to its own listener. Suppose the speaker has access to a small **population** of listener models $\{L_0^{(1)}, \dots, L_0^{(K)}\}$, but the true listener is none of them. Can it plan in a way that generalizes?

We add a second training listener, the **LAION-2B ViT-B/32** CLIP. It has the same architecture as OpenAI's ViT-B/32 but was trained on a different dataset. ViT-B/16 stays **held out** for evaluation only. We also add the 12 two-word utterances, such as *red circle*, at cost $\lambda$ per utterance. Two-word utterances are nearly always unambiguous, so the speaker now trades off **informativity** against **brevity**.

## 4.1 Population speakers (6 points, autograded)

Implement `population_speaker(log_L0_list, target, cost, mode)`, which returns the utterance index maximizing

- `mode="mean"`: $\frac{1}{K}\sum_k \log L_0^{(k)}(\text{target}\mid u) - C(u)$, i.e. the geometric-mean listener;
- `mode="min"`: $\min_k \log L_0^{(k)}(\text{target}\mid u) - C(u)$, i.e. the worst-case listener.

Each `log_L0_list[k]` is a `[U, C]` array, for example the output of `context_log_L0`. `cost` is a `[U]` array.

In [ ]:
# GRADED: population_speaker
import numpy as np


def population_speaker(log_L0_list, target, cost, mode="mean"):
    """Returns the index (int) of the utterance the population speaker chooses for `target`."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
clip_laion, proc_laion = load_clip("laion/CLIP-ViT-B-32-laion2B-s34B-b79K")
S_laion = score_matrix(clip_laion, proc_laion, shape_images, prompts)
del clip_laion

ALL_UTTS = np.arange(len(UTTERANCES))
def cost_vec(lam):
    return np.r_[np.zeros(7), np.full(12, lam)]

def make_pop_speaker(score_list, lam, mode):
    return lambda ctx: population_speaker([context_log_L0(S[ctx]) for S in score_list], 0, cost_vec(lam), mode)

def make_S1_cost(S, lam):
    return make_S1(S, utts=ALL_UTTS, cost=cost_vec(lam))

L0_b16_all = make_L0(S_b16, ALL_UTTS)
LAM = 0.5
speakers_4 = {
    "S1(B/32)":              make_S1_cost(S_b32, LAM),
    "S1(LAION)":             make_S1_cost(S_laion, LAM),
    "pop-mean(B/32+LAION)":  make_pop_speaker([S_b32, S_laion], LAM, "mean"),
    "pop-min(B/32+LAION)":   make_pop_speaker([S_b32, S_laion], LAM, "min"),
    "S1(B/16) [oracle]":     make_S1_cost(S_b16, LAM),
}
rows = {}
for name, spk in speakers_4.items():
    utts = [spk(ctx) for ctx in CONTEXTS["hard"]]
    rows[name] = {"held-out L0(B/16) acc": play(spk, L0_b16_all, CONTEXTS["hard"]),
                  "% two-word": np.mean(np.array(utts) >= 7)}
display(pd.DataFrame(rows).T.round(3))

## ★ 4.2 Better, or just longer? (6 points, written)

The population speakers may succeed more often simply because they use more two-word utterances. To compare speakers fairly, sweep $\lambda$ and plot **held-out accuracy against the fraction of two-word utterances** for `S1(B/32)`, `pop-mean`, and `pop-min` in hard contexts. Each speaker traces an accuracy–verbosity curve. Put all three curves on one figure, with labeled axes and a legend.

Use 400 hard contexts to keep it fast. A suggested λ grid is `[0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5]`.

In [ ]:
# YOUR CODE HERE (not autograded: report what you find in the written answer below)

## ★ 4.3 Analysis (8 points, written)

(a) Based on the table in 4.1 and your figure in 4.2, is the population speaker a *better* communicator, or just a more *verbose* one? Support your answer with specific numbers. Compare points with similar verbosity.

(b) Explain the behavior of `pop-min` mechanistically. In which contexts does taking the worst case over listeners change the speaker's choice?

(c) Propose one concrete method that should move the curve itself, not just the position along it, and say how you would test it. You do not need to implement it. Example directions include calibrating each listener before combining them, adapting to the partner from a few rounds of feedback, and a larger or more diverse population.

**Your answer (Q4.3):**

*Write your answer here.*

---
## Before you submit

1. *Runtime → Restart and run all.* Make sure every cell runs without errors. EECS 183 students only need Parts 1–3 to run, and may delete the Part 4 cells.
2. Check that each `# GRADED:` cell still begins with its original header line and defines the required functions.
3. Export `HW6.pdf` and check that every plot rendered.
4. Upload `HW6.ipynb` and `HW6.pdf` to Gradescope **A6**, or upload the `submission.zip` built by `python3 make_submission.py`.